In [ ]:
# =========================
# Básicas
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# =========================
# Astronomía / Light curves
# =========================
from astropy.table import Table
from wotan import flatten
import lightkurve as lk
# =========================
# Modelado (binarias / exoplanetas)
# =========================
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import starry
import theano.tensor as tt

# =========================
# Visualización avanzada (opcional)
# =========================
from mpl_toolkits import mplot3d
from matplotlib import animation
from matplotlib.ticker import ScalarFormatter
from IPython.display import HTML

# =========================
# Utilidades
# =========================
from scipy.stats import median_abs_deviation
from tqdm.notebook import tqdm
from corner import corner
import dataframe_image as dfi

In [ ]:
# =========================
# Configuracion de rutas locales
# =========================
RESULTS_DIR = r"/home/luciam/Documents/starry/Resultados_Orion"


In [ ]:
np.random.seed(12)
starry.config.lazy = False
starry.config.quiet = True

In [ ]:

# =========================
# 1. Cargar datos
# =========================
file_path = "/home/luciam/Documents/starry/Resultados_Orion/Light_Curve_Files/tic 138829403_S_6_LC.txt"

data = pd.read_csv(file_path, sep=r"\s+")
data = data.rename(columns={"#Time": "Time"})

time = np.asarray(data["Time"], dtype=float)
flux = np.asarray(data["Flux_CBV_SS"], dtype=float)
sigma_real = np.asarray(data["Flux_err_SS"], dtype=float)
#Buscar agregar un filtro antes de meterlo a la funcion, modificar aparte flux para hacer el detrending antes de meterlo a la funcion
# =========================
# 2. Detrending con WOTAN
# =========================
flat_flux, trend_flux = flatten(
    time,
    flux,
    window_length=0.55,
    break_tolerance=1.5,
    method="biweight",
    return_trend=True
)


# =========================
# 3. Plot original + trend
# =========================
plt.figure(figsize=(10,4))

plt.scatter(time, flux, s=5, color='k', label="Original")
plt.plot(time, trend_flux, color='r', lw=2, label="Trend")

plt.xlabel('Time MJD [days]')
plt.ylabel('Flux')
plt.legend()
plt.title("Wotan Detrending")
plt.tight_layout()
plt.show()


# =========================
# 4. Plot flatten (el importante)
# =========================
plt.figure(figsize=(10,4))

plt.scatter(time, flat_flux, s=4.5, color='b')

plt.xlabel('Time MJD [days]')
plt.ylabel('Normalised Flux')
plt.title("Flattened Light Curve (Wotan)")
plt.tight_layout()
plt.show()


# =========================
# 5. Guardar resultado
# =========================
df = pd.DataFrame({
    "time": time,
    "flat_flux": flat_flux,
    "trend_flux": trend_flux
})

df.to_csv("tic_138829403_trending.csv", index=False)

# Starry

In [ ]:
from astroquery.mast import Catalogs

tic_id = "TIC 138829403"
Label = "TIC 138829403"
cat = Catalogs.query_object(tic_id, catalog="tic")
 
cat['ID', 'Teff', 'logg', 'mass', 'rad', 'Tmag', 'GAIAmag']
r_A      = float(cat['rad'][0])#  radius in R_sun
m_A      = float(cat['mass'][0])
teff = float(cat['Teff'][0])
Tmag = float(cat["Tmag"][0])
print(r_A)
print(m_A)
print(teff)
print(Tmag)

In [ ]:
# =========================
# 4b. Priors desde CSV (Orion) + estimacion de R_B a partir de R_A (MAST)
# =========================
r_A = 1.63226
teff = 5553.7
priors_path = "/home/luciam/Documents/starry/Orion_yceb/priors.csv"
priors_df = pd.read_csv(priors_path)

tic_num = int(tic_id.replace("TIC", "").strip())
rows = priors_df[priors_df["TIC"] == tic_num]

if rows.empty:
    print(f"No se encontraron priors para {tic_id} (TIC {tic_num}) en {priors_path}")
    print("Revisa que el TIC del notebook coincida con el TIC del CSV de priors.")
else:
    print(f"Priors encontrados para {tic_id} en {len(rows)} sector(es):\n")
    display(rows)

    for _, row in rows.iterrows():
        print(f"--- Sector {row['Sector']} ---")
        print(f"  t0              = {row['t0']}")
        print(f"  Duration        = {row['Duration']}")
        print(f"  P_bls           = {row['P_bls']}")
        print(f"  P_orb           = {row['P_orb']}")
        print(f"  P_rot           = {row['P_rot']}")
        print(f"  Depth_primary   = {row['Depth_primary']} +/- {row['Depth_primary_err']}")
        print(f"  Depth_secondary = {row['Depth_secondary']} +/- {row['Depth_secondary_err']}")

        # Estimacion de R_B a partir de la profundidad del eclipse primario y R_A (MAST).
        # Aproximacion de eclipse total/anular: depth ~ (R_B / R_A)^2  =>  R_B ~ R_A * sqrt(depth)
        # Es solo un prior inicial grueso -- no reemplaza el ajuste con Starry.
        depth_p = row["Depth_primary"]
        depth_p_err = row["Depth_primary_err"]
        if pd.notna(depth_p) and depth_p > 0:
            r_B_est = r_A * np.sqrt(depth_p)
            if pd.notna(depth_p_err):
                # propagacion simple de error: sigma_R_B/R_B = 0.5 * sigma_depth/depth
                r_B_est_err = r_B_est * 0.5 * (depth_p_err / depth_p)
                print(f"  R_B estimado    = {r_B_est:.4f} +/- {r_B_est_err:.4f} R_sun (R_A={r_A:.4f} R_sun)")
            else:
                print(f"  R_B estimado    = {r_B_est:.4f} R_sun (R_A={r_A:.4f} R_sun)")
        print()


In [ ]:
# =========================
# Priors
# =========================

inc_A, sd_inc_A = 88.0, 1
amp_A, sd_amp_A = 0.998, 0.001
r_A, sd_r_A = 1.63226, 0.1664
m_A, sd_m_A = 2.775, 0.291
u_A = [0.3, 0.2]

inc_B, sd_inc_B = 88.0, 1
amp_B, sd_amp_B = 0.002, 0.001
r_B, sd_r_B = 0.3125, 0.0146
m_B, sd_m_B = .98 , 0.09
porb_B, sd_porb_B = 2.903, 0.01
t0_B, sd_t0_B = 1469.855, 0.12
ecc_B, sd_ecc_B = 0.085, 0.01
w_B, sd_w_B = 215.03, 2
u_B = [0.3, 0.2]

prot_s = 0

In [ ]:
t_real = time
flux_real = flat_flux
sigma_sd = 1.4826*median_abs_deviation(flux_real)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12,5))
ax.plot(t_real, flux_real, ".k", alpha=0.6, ms=5)
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
map = starry.Map(ydeg=5)
A_y = np.array(map.y[1:])

map.reset()
B_y = np.array(map.y[1:])

In [ ]:
A = dict(
    ydeg   = 5,  # degree of the map
    udeg   = 2,  # degree of the limb darkening
    inc    = inc_A,  # inclination in degrees
    amp    = amp_A,  # amplitude (a value prop. to luminosity)
    r      = r_A, #  radius in R_sun
    m      = m_A,  # mass in M_sun
    prot   = prot_s,  # rotational period in days
    u      = u_A,  # limb darkening coefficients
    y      = A_y,  # the spherical harmonic coefficients
)
B = dict(
    ydeg   = 5,  # degree of the map
    udeg   = 2,  # degree of the limb darkening
    inc    = inc_B,  # inclination in degrees
    amp    = amp_B,  # amplitude (a value prop. to luminosity)
    r      = r_B,  #  radius in R_sun
    m      = m_B,  #  mass in M_sun
    prot   = prot_s,  # rotational period in days
    porb   = porb_B,  # orbital period in days
    t0     = t0_B,  # reference time in days (when it transits A)
    ecc    = ecc_B,
    w      = w_B,  # longitude of pericenter in degrees
    u      = u_B,  # limb darkening coefficients
    y      = B_y,  # the spherical harmonic coefficients
)

In [ ]:
print(A["y"])
print(type(A["y"]))
print(np.asarray(A["y"]).dtype)

In [ ]:
pri = starry.Primary(
    starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A["inc"], amp=A["amp"]),
    r=A["r"],
    m=A["m"],
    prot=A["prot"],
)
pri.map[1:] = A["u"]
pri.map[1:, :] = A["y"]

sec = starry.Secondary(
    starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B["inc"], amp=B["amp"]),
    r=B["r"],
    m=B["m"],
    porb=B["porb"],
    prot=B["prot"],
    t0=B["t0"],
    inc=B["inc"],
    ecc=B["ecc"],
    w=B['w'],
)
sec.map[1:] = B["u"]
sec.map[1:, :] = B["y"]

In [ ]:
sys = starry.System(pri, sec)

In [ ]:
t = t_real
flux_true = sys.flux(t_real)
#sigma = sigma_real
# flux = flux_real + sigma_real * np.random.randn(len(t))
flux = flux_real + sigma_sd * np.random.randn(len(t))

In [ ]:
fig, ax = plt.subplots(1, figsize=(18, 9))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=10)
ax.plot(t_real, flux_true, lw=2)
# ax.plot(t_real, flux_real, ".k", ms=4, color= 'k' ,label="Sector 06")
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

# ax.set_xlim(2175 , 2181);

In [ ]:
np.savez(Label+".npz", A=A, B=B, t=t_real, flux=flux, sigma=sigma_real)

np.random.seed(12)
starry.config.lazy = True
starry.config.quiet = True

data = np.load(Label+".npz", allow_pickle=True)
A = data["A"].item()
B = data["B"].item()
t = data["t"]
flux = data["flux"]
sigma = data["sigma"]

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 5))
ax.plot(t, flux_real, "b.", alpha=0.5, ms=5)
ax.plot(t, flux, "k.", alpha=0.5, ms=5)
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
print("A['u']:", A["u"], type(A["u"]), np.asarray(A["u"]).shape)
print("B['u']:", B["u"], type(B["u"]), np.asarray(B["u"]).shape)

In [ ]:
with pm.Model() as model:
    PositiveNormal = pm.Bound(pm.Normal, lower=0.0)

    # Primary
    A_inc = pm.Normal("A_inc", mu = inc_A, sd = sd_inc_A, testval = inc_A)
    A_amp = pm.Normal("A_amp", mu = amp_A, sd = sd_amp_A, testval= amp_A)
    A_r = PositiveNormal("A_r", mu = r_A, sd = sd_r_A, testval = r_A)
    A_m = PositiveNormal("A_m", mu = m_A, sd = sd_m_A, testval = m_A)
    A_prot = prot_s
    u_A = xo.distributions.QuadLimbDark("u_A", testval=np.array(A['u']))
    pri = starry.Primary(
        starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A_inc, amp=A_amp),
        r=A_r,
        m=A_m,
        prot=A_prot,
    )
    pri.map[1] = A["u"][0]
    pri.map[2] = A["u"][1]

    # Secondary
    B_inc = pm.Normal("B_inc", mu = inc_B, sd = sd_inc_B, testval = inc_B)
    B_amp = pm.Normal("B_amp", mu = amp_B, sd = sd_amp_B, testval = amp_B)
    B_r = PositiveNormal("B_r", mu = r_B, sd = sd_r_B, testval = r_B)
    B_m = PositiveNormal("B_m", mu = m_B, sd = sd_m_B, testval = m_B)
    B_prot = prot_s
    B_porb = PositiveNormal("B_porb", mu = porb_B, sd = sd_porb_B, testval = porb_B)
    B_t0 = pm.Normal("B_t0", mu = t0_B, sd = sd_t0_B, testval = t0_B)
    u_B = xo.distributions.QuadLimbDark("u_B", testval=np.array(B['u']))
    B_ecc = pm.TruncatedNormal("B_ecc", mu = ecc_B, sigma = sd_ecc_B, lower = 0, upper = 1)
    #B_ecc = pm.Uniform("B_ecc", 0, 1, initval=ecc_B)
    B_w = pm.Normal("B_w", mu = w_B, sd = sd_w_B, testval = w_B)#pm.Uniform("B_w", 0, 360, initval = w_B)
    #B_Omega = pm.Uniform("B_Omega", 0, 360, initval=Omega_B)
    sec = starry.Secondary(
        starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B_inc, amp=B_amp),
        r=B_r,
        m=B_m,
        porb=B_porb,
        prot=B_prot,
        t0=B_t0,
        inc=B_inc,
        ecc=B_ecc,
        w=B_w,
        #Omega=B_Omega
    )
    sec.map[1] = B["u"][0]
    sec.map[2] = B["u"][1]

    # System
    sys = starry.System(pri, sec)

    # Compute the flux
    flux_model = sys.flux(t=t)

    # Track some values for plotting later
    pm.Deterministic("flux_model", flux_model)

    # Save our initial guess
    flux_model_guess = pmx.eval_in_model(flux_model)

    # The likelihood function assuming known Gaussian uncertainty
    pm.Normal("obs", mu=flux_model, sd=sigma_sd, observed=flux_real)


In [ ]:
with model:
    map_soln = pmx.optimize(start=model.test_point)

In [ ]:
map_soln

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, flux, "k.", alpha=0.3, ms=2, label="data")
plt.plot(t, flux_model_guess, "C1--", lw=1, alpha=0.5, label="Initial")
plt.plot(
    t, pmx.eval_in_model(flux_model, map_soln, model=model), "C1-", label="MAP", lw=1
)
plt.legend(fontsize=10, numpoints=5)
plt.xlabel("time [days]", fontsize=24)
plt.ylabel("relative flux", fontsize=24);

In [ ]:
with model:
    trace = pmx.sample(
        tune=1500, draws=1500, start=map_soln, cores=5, chains=4, target_accept=0.9, return_inferencedata=True,
    )

In [ ]:
var_names_A = ["A_m", "A_r", "A_inc", "u_A", 'A_amp']
var_names_B = ["B_m", "B_r", "B_porb", "B_inc", "B_t0", "B_ecc", "B_w", 'u_B', 'B_amp']
display(pm.summary(trace, var_names=var_names_A + var_names_B))

# Obtener el resumen
summary = pm.summary(trace, var_names=var_names_A + var_names_B)

# Guardar en un archivo CSV
summary.to_csv(Label+'_Summary.csv', sep=',')

In [ ]:
figure = corner(trace, var_names=var_names_A, show_titles=True,use_math_text=True,label_kwargs={"fontsize": 14})

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=12)


figure.savefig(Label+'_A.png',dpi=600,pad_inches=0.3,bbox_inches='tight')

In [ ]:
figure = corner(trace, var_names=var_names_B, show_titles=True,use_math_text=True,label_kwargs={"fontsize": 14})
for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=12)


figure.savefig(Label+'_B.png',dpi=600, pad_inches=0.3, bbox_inches='tight')

In [ ]:
curve = trace.posterior["flux_model"].values

# Percentiles 16, 50 y 84
q16, q50, q84 = np.percentile(
    curve,
    [16, 50, 84],
    axis=(0, 1)
)

# ============================================
# Guardar curvas
# ============================================

Curva_Q50 = lk.LightCurve(time=t, flux=q50)
Curva_Q16 = lk.LightCurve(time=t, flux=q16)
Curva_Q84 = lk.LightCurve(time=t, flux=q84)

with open(Label + "_q50.csv", "w") as text_file:
    text_file.write(Curva_Q50.to_csv())

with open(Label + "_q16.csv", "w") as text_file:
    text_file.write(Curva_Q16.to_csv())

with open(Label + "_q84.csv", "w") as text_file:
    text_file.write(Curva_Q84.to_csv())


# ============================================
# Carpeta de figuras
# ============================================

Figures_dir = "Figures"
os.makedirs(Figures_dir, exist_ok=True)


# ============================================
# Plot
# ============================================

fig, ax = plt.subplots(figsize=(12, 5))

# Banda de incertidumbre 16--84 %
ax.fill_between(
    t,
    q16,
    q84,
    alpha=0.3,
    label=r"16--84 percentile"
)

# Modelo mediano
ax.plot(
    t,
    q50,
    linewidth=2,
    label="Median model"
)

# Datos observados
ax.scatter(
    t,
    flux,
    s=8,
    alpha=0.5,
    label="Observed"
)

# ============================================
# Título
# ============================================

ax.set_title(
    f"{Label}",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Time [d]")
ax.set_ylabel("Normalized flux")

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()


# ============================================
# Guardar figura
# ============================================

fig_path = os.path.join(
    Figures_dir,
    f"{Label}_model.png"
)

plt.savefig(
    fig_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Figura guardada en: {fig_path}")

In [ ]:
# Plot the results
curve = trace.posterior["flux_model"].values
q16, q50, q84 = np.percentile(curve, [16, 50, 84], axis=(0, 1))

Curva_Q50 = lk.LightCurve(time=t, flux=q50)
Curva_Q16 = lk.LightCurve(time=t, flux=q16)
Curva_Q84 = lk.LightCurve(time=t, flux=q84)

strm_1 = Curva_Q50.to_csv()
with open(Label+"_q50.csv", "w") as text_file:
    text_file.write(strm_1)

strm_2 = Curva_Q16.to_csv()
with open(Label+"_q16.csv", "w") as text_file:
    text_file.write(strm_2)

strm_3 = Curva_Q84.to_csv()
with open(Label+"_q84.csv", "w") as text_file:
    text_file.write(strm_2)

Curva_Q50

In [ ]:
fig, ax = plt.subplots(1, figsize=(18, 9))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=10)
ax.plot(Curva_Q50['time'].value, Curva_Q50['flux'].value, lw=2)
# ax.plot(t_real, flux_real, ".k", ms=4, color= 'k' ,label="Sector 06")
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
import pandas as pd

inf_pos = {"A_m": trace.posterior["A_m"].values.reshape(-1), "A_r": trace.posterior["A_r"].values.reshape(-1),
           "A_inc": trace.posterior["A_inc"].values.reshape(-1), "A_amp": trace.posterior["A_amp"].values.reshape(-1),
           'A_u0':trace.posterior["u_A"][:,:,0].values.reshape(-1), 'A_u1':trace.posterior["u_A"][:,:,1].values.reshape(-1),
           "B_m": trace.posterior["B_m"].values.reshape(-1), "B_r": trace.posterior["B_r"].values.reshape(-1),
           "B_porb": trace.posterior["B_porb"].values.reshape(-1), "B_inc": trace.posterior["B_inc"].values.reshape(-1),
           "B_t0": trace.posterior["B_t0"].values.reshape(-1), "B_ecc": trace.posterior["B_ecc"].values.reshape(-1),
           "B_w": trace.posterior["B_w"].values.reshape(-1), "B_amp": trace.posterior["B_amp"].values.reshape(-1),
           'B_u0':trace.posterior["u_B"][:,:,0].values.reshape(-1), 'B_u1':trace.posterior["u_B"][:,:,1].values.reshape(-1),}
print(inf_pos["A_m"].shape)
df_pos = pd.DataFrame(inf_pos)

df_pos.to_csv(Label+"_Posteriors.csv", sep=',')

# Calcular el promedio de cada columna y mostrarlo en pantalla
column_means = df_pos.mean()
print(column_means)

In [ ]:
df_pos

In [ ]:
from astropy.io import ascii
starry.config.lazy = False
starry.config.quiet = True

data = ascii.read('data.txt', format='fast_no_header')
file = data[0][0]
path = data[2][0]

#system = file[-34:-15]
system = file[-32:-15]

print(system)

Pos = pd.read_csv(data[0][0])

Q16 = pd.read_csv(data[2][0])  
Q50 = pd.read_csv(data[3][0])  
Q84 = pd.read_csv(data[4][0])  


In [ ]:
def phase_fold(t, y, period, t0):
    phases=np.remainder(t-t0, period) / period
    phases=np.concatenate((phases,phases+1))
    y=np.concatenate((y,y))
    return phases, y

In [ ]:

from pathlib import Path

system_name = Path(system).name

print(system_name)

In [ ]:
import corner
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np


# ============================================================
# Parámetros visuales del corner plot
# ============================================================

CORNER_KWARGS = dict(
    # Suavizado
    smooth=0.7,

    # Etiquetas de los ejes
    label_kwargs={
        "fontsize": 14,
        "labelpad": 8
    },

    # Valores de los parámetros
    title_kwargs={
        "fontsize": 11,
        "pad": 10
    },

    # Intervalos de confianza
    quantiles=[0.16, 0.50, 0.84],

    # Distribuciones
    plot_density=True,
    plot_datapoints=True,

    # Contornos
    fill_contours=True,

    # Mostrar mediana y valores
    show_titles=True,

    # Número de ticks
    max_n_ticks=4,

    # Formato de los valores superiores
    title_fmt=".3f",

    # Notación matemática
    use_math_text=True,

    # Apariencia
    color="#4C6A4A",

    # Transparencia de los puntos
    hist_kwargs={
        "density": True,
        "alpha": 0.75,
        "linewidth": 1.2
    },

    # Apariencia de los datos
    data_kwargs={
        "ms": 2.0,
        "alpha": 0.20
    },

    # Contornos
    contour_kwargs={
        "linewidths": 1.2
    }
)


# ============================================================
# Parámetros
# ============================================================

array_a = Pos[["A_m", "A_r"]].values


# ============================================================
# Crear figura
# ============================================================

figure = corner.corner(
    array_a,
    labels=[
        r"$M_A\ [M_\odot]$",
        r"$R_A\ [R_\odot]$"
    ],
    fig=plt.figure(figsize=(6.2, 6.2)),
    **CORNER_KWARGS
)


# ============================================================
# Personalizar ejes
# ============================================================

axes = figure.get_axes()

for ax in axes:

    # Tamaño de los ticks
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=10,
        length=4,
        width=0.8
    )

    # Separación respecto al eje
    ax.tick_params(
        axis="x",
        which="both",
        pad=3
    )

    ax.tick_params(
        axis="y",
        which="both",
        pad=3
    )

    # Grosor de los bordes
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


# ============================================================
# Formato de los ticks
# ============================================================

for ax in axes:

    ax.xaxis.set_major_formatter(
        ticker.FormatStrFormatter("%.2f")
    )

    ax.yaxis.set_major_formatter(
        ticker.FormatStrFormatter("%.2f")
    )


# ============================================================
# Título general
# ============================================================

figure.suptitle(
    rf"TIC {system_name}",
    fontsize=16,
    fontweight="bold",
    y=1.02
)


# ============================================================
# Ajuste final
# ============================================================

figure.subplots_adjust(
    top=0.92,
    bottom=0.12,
    left=0.12,
    right=0.97,
    hspace=0.05,
    wspace=0.05
)


# ============================================================
# Guardar
# ============================================================

figure.savefig(
    f"Corner_{system_name}_A.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import corner


# ============================================================
# Parámetros visuales del corner plot
# ============================================================

CORNER_KWARGS = dict(
    # Suavizado
    smooth=0.7,

    # Etiquetas de los ejes
    label_kwargs={
        "fontsize": 14,
        "labelpad": 10
    },

    # Valores de los parámetros
    title_kwargs={
        "fontsize": 11,
        "pad": 12
    },

    # Intervalos de confianza
    quantiles=[0.16, 0.50, 0.84],

    # Distribuciones
    plot_density=True,
    plot_datapoints=True,

    # Contornos
    fill_contours=True,

    # Mostrar mediana y valores
    show_titles=True,

    # Número de ticks
    max_n_ticks=4,

    # Formato de los valores superiores
    title_fmt=".3f",

    # Notación matemática
    use_math_text=True,

    # Color principal
    color="#4C6A4A",

    # Apariencia de las distribuciones 1D
    hist_kwargs={
        "density": True,
        "alpha": 0.75,
        "linewidth": 1.2
    },

    # Apariencia de los puntos
    data_kwargs={
        "ms": 1.8,
        "alpha": 0.18
    },

    # Apariencia de los contornos
    contour_kwargs={
        "linewidths": 1.2
    }
)


# ============================================================
# Parámetros del componente B
# ============================================================

array_B = Pos[
    ["B_m", "B_r", "B_inc", "B_ecc", "B_w"]
].values


# ============================================================
# Constantes físicas
# ============================================================

G       = 6.6742e-11
M_sun   = 1.99e30
sec_day = 86400
AU      = 150e9


# ============================================================
# Cálculo del semieje mayor
# ============================================================

mass_1 = Pos["A_m"].values
P = Pos["B_porb"].values * sec_day

a = (
    (
        G
        * (mass_1 + array_B[:, 0])
        * M_sun
        * P**2
    )
    / (4 * np.pi**2)
)**(1 / 3) / AU


# ============================================================
# Construir array final
#
# Orden:
# M_B, R_B, i_B, a, e_B, omega_B
# ============================================================

new_array = np.column_stack((
    array_B[:, 0],   # M_B
    array_B[:, 1],   # R_B
    array_B[:, 2],   # i_B
    a,               # a
    array_B[:, 3],   # e_B
    array_B[:, 4]    # omega_B
))


# ============================================================
# Crear corner plot
# ============================================================

figure = corner.corner(
    new_array,

    labels=[
        r"$M_B\ [M_\odot]$",
        r"$R_B\ [R_\odot]$",
        r"$i_B\ [^\circ]$",
        r"$a\ [AU]$",
        r"$e_B$",
        r"$\omega_B\ [^\circ]$"
    ],

    # Figura más grande para los 6 parámetros
    fig=plt.figure(figsize=(11.5, 11.5)),

    **CORNER_KWARGS
)


# ============================================================
# Personalizar ejes
# ============================================================

axes = figure.get_axes()

for ax in axes:

    # --------------------------------------------------------
    # Ticks
    # --------------------------------------------------------

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=9,
        length=4,
        width=0.8
    )

    # Separación de las etiquetas de los ticks
    ax.tick_params(
        axis="x",
        which="both",
        pad=4
    )

    ax.tick_params(
        axis="y",
        which="both",
        pad=4
    )

    # --------------------------------------------------------
    # Bordes
    # --------------------------------------------------------

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


# ============================================================
# Formato de los ticks
# ============================================================

for ax in axes:

    ax.xaxis.set_major_formatter(
        ticker.FormatStrFormatter("%.2f")
    )

    ax.yaxis.set_major_formatter(
        ticker.FormatStrFormatter("%.2f")
    )


# ============================================================
# Título general
# ============================================================

figure.suptitle(
    rf"TIC {system_name}",
    fontsize=16,
    fontweight="bold",
    y=0.995
)


# ============================================================
# Ajuste final de la figura
# ============================================================

figure.subplots_adjust(
    top=0.94,
    bottom=0.09,
    left=0.09,
    right=0.97,
    hspace=0.08,
    wspace=0.08
)


# ============================================================
# Guardar
# ============================================================

figure.savefig(
    f"Corner_{system_name}_B.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


# ============================================================
# Mostrar
# ============================================================

plt.show()

In [ ]:
# Get the posterior median orbital parameters
M_a_16, M_a_50, M_a_84         = np.percentile(Pos['A_m'], [16, 50, 84])
R_a_16, R_a_50, R_a_84         = np.percentile(Pos['A_r'], [16, 50, 84])
i_a_16, i_a_50, i_a_84         = np.percentile(Pos['A_inc'], [16, 50, 84])
amp_a_16, amp_a_50, amp_a_84   = np.percentile(Pos['A_amp'], [16, 50, 84])
u0_a_16, u0_a_50, u0_a_84      = np.percentile(Pos['A_u0'], [16, 50, 84])
u1_a_16, u1_a_50, u1_a_84      = np.percentile(Pos['A_u1'], [16, 50, 84])

M_b_16, M_b_50, M_b_84         = np.percentile(Pos['B_m'], [16, 50, 84])
R_b_16, R_b_50, R_b_84         = np.percentile(Pos['B_r'], [16, 50, 84])
i_b_16, i_b_50, i_b_84         = np.percentile(Pos['B_inc'], [16, 50, 84])
p_b_16, p_b_50, p_b_84         = np.percentile(Pos['B_porb'], [16, 50, 84])
t0_b_16, t0_b_50, t0_b_84      = np.percentile(Pos['B_t0'], [16, 50, 84])
e_b_16, e_b_50, e_b_84         = np.percentile(Pos['B_ecc'], [16, 50, 84])
w_b_16, w_b_50, w_b_84         = np.percentile(Pos['B_w'], [16, 50, 84])
amp_b_16, amp_b_50, amp_b_84   = np.percentile(Pos['B_amp'], [16, 50, 84])
u0_b_16, u0_b_50, u0_b_84      = np.percentile(Pos['B_u0'], [16, 50, 84])
u1_b_16, u1_b_50, u1_b_84      = np.percentile(Pos['B_u1'], [16, 50, 84])

a_b_16, a_b_50, a_b_84         = np.percentile(a, [16, 50, 84])

print(amp_a_50, amp_b_50)

In [ ]:
print(system)
print('Table_'+system+'.png')

In [ ]:
# ============================================================
# 1. Función para formatear los resultados
# ============================================================

def format_parameter(q16, q50, q84, decimals=3):
    """
    Formato:
        x^{+upper}_{-lower}

    donde:
        lower = q50 - q16
        upper = q84 - q50
    """

    lower = q50 - q16
    upper = q84 - q50

    return (
        rf"${q50:.{decimals}f}"
        rf"^{{+{upper:.{decimals}f}}}"
        rf"_{{-{lower:.{decimals}f}}}$"
    )


# ============================================================
# 2. Construir la tabla
# ============================================================

data = {

    'Parameter': [
        r'$M_A$',
        r'$R_A$',
        r'$u_{0,A}$',
        r'$u_{1,A}$',
        r'$M_B$',
        r'$R_B$',
        r'$i$',
        r'$P_{\mathrm{orb}}$',
        r'$a$',
        r'$t_0$',
        r'$e$',
        r'$\omega$',
        r'$u_{0,B}$',
        r'$u_{1,B}$'
    ],

    'Value': [

        format_parameter(
            M_a_16, M_a_50, M_a_84,
            decimals=3
        ),

        format_parameter(
            R_a_16, R_a_50, R_a_84,
            decimals=3
        ),

        format_parameter(
            u0_a_16, u0_a_50, u0_a_84,
            decimals=3
        ),

        format_parameter(
            u1_a_16, u1_a_50, u1_a_84,
            decimals=3
        ),

        format_parameter(
            M_b_16, M_b_50, M_b_84,
            decimals=3
        ),

        format_parameter(
            R_b_16, R_b_50, R_b_84,
            decimals=3
        ),

        format_parameter(
            i_b_16, i_b_50, i_b_84,
            decimals=3
        ),

        format_parameter(
            p_b_16, p_b_50, p_b_84,
            decimals=5
        ),

        format_parameter(
            a_b_16, a_b_50, a_b_84,
            decimals=3
        ),

        format_parameter(
            t0_b_16, t0_b_50, t0_b_84,
            decimals=5
        ),

        format_parameter(
            e_b_16, e_b_50, e_b_84,
            decimals=4
        ),

        format_parameter(
            w_b_16, w_b_50, w_b_84,
            decimals=3
        ),

        format_parameter(
            u0_b_16, u0_b_50, u0_b_84,
            decimals=3
        ),

        format_parameter(
            u1_b_16, u1_b_50, u1_b_84,
            decimals=3
        )
    ],

    'Unit': [
        r'$M_\odot$',
        r'$R_\odot$',
        '',
        '',
        r'$M_\odot$',
        r'$R_\odot$',
        r'$\mathrm{deg}$',
        r'$\mathrm{d}$',
        r'$R_\odot$',
        r'$\mathrm{d}$',
        '',
        r'$\mathrm{rad}$',
        '',
        ''
    ]
}


# ============================================================
# 3. Crear DataFrame
# ============================================================

Final_info_B = pd.DataFrame(data)

import dataframe_image as dfi
# ============================================================
# 4. Crear carpeta de salida
# ============================================================

output_dir = "Tables"
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 5. Nombre del archivo
# ============================================================

system_name = system.split('/')[-1]

filename = os.path.join(
    output_dir,
    f"{system_name}_parameters.png"
)


# ============================================================
# 6. Mostrar la tabla en el notebook
# ============================================================

display(Final_info_B)


# ============================================================
# 7. Exportar como imagen
# ============================================================

dfi.export(
    Final_info_B,
    filename,
    table_conversion='matplotlib',
    dpi=300
)


print(f"\nTabla guardada en:")
print(filename)

In [ ]:
A = starry.Primary(starry.Map(ydeg=3, udeg=2, amp=amp_a_50), r=R_a_50, m=M_a_50)
A.map[1:] = [u0_a_50, u1_a_50]

A.map.show()

In [ ]:
b = starry.Secondary(
    starry.Map(ydeg=3, udeg=2, inc=i_b_50, amp=amp_b_50),
    r=R_b_50,
    m=M_b_50,
    porb=p_b_50,
    prot=1.0,
    t0=t0_b_50,
    inc=i_b_50,
    ecc=e_b_50,
    w=w_b_50,
)
b.map[1:] = [u0_b_50, u1_b_50]

b.map.show()

In [ ]:
sys = starry.System(A, b)
sys

In [ ]:
npts = 200
t = np.linspace(t0_b_50 - p_b_50/2, t0_b_50 + p_b_50/2, npts)
x, y, z = sys.position(t)
x = x - x[0]
y = y - y[0]
z = z - z[0]


In [ ]:
res = 300
theta_sec = [360.0 / sec.prot * (t - sec.t0) - sec.theta0 for sec in sys.secondaries]
img = np.array(
    [np.tile(sys.primary.map.render(res=300), (npts, 1, 1))]
    + [
        sec.map.render(theta=theta_sec[i], res=res)
        for i, sec in enumerate(sys.secondaries)
    ]
)

In [ ]:
flux = sys.flux(t)

t_fold = ((t - t0_b_50)/p_b_50)

In [ ]:
im0 = sys.show(t, window_pad=5, figsize=(5, 5))

In [ ]:
# Set up the plot
fig, ax = plt.subplots(1, figsize=(6.5, 7))
ax_xz = fig.add_axes([0.275, 0.8, 0.2, 0.2])
ax_xz.annotate(
    "Top", fontsize=12, xy=(0, 0), xycoords="axes fraction", ha="left", va="bottom"
)
ax_zy = fig.add_axes([0.525, 0.8, 0.2, 0.2])
ax_zy.annotate(
    "Side", fontsize=12, xy=(0, 0), xycoords="axes fraction", ha="left", va="bottom"
)
ax_lc = fig.add_axes([0.125, 0.1, 0.775, 0.2])

xz = [None] + [None for sec in sys.secondaries]
xy = [None] + [None for sec in sys.secondaries]
zy = [None] + [None for sec in sys.secondaries]
circ = [None] + [None for sec in sys.secondaries]
maps = [sys.primary.map] + [sec.map for sec in sys.secondaries]
radii = np.array([sys.primary.r] + [sec.r for sec in sys.secondaries])

for axis, arrs in zip([ax, ax_xz, ax_zy], [(x, y), (x, z), (z, y)]):
    axis.axis("off")
    R = 1.2 * max(-np.min(arrs), np.max(arrs))
    axis.set_xlim(-R, R)
    axis.set_ylim(-R, R)

# Plot the light curve
ax_lc.plot(t_fold, flux, "k-")
(lc,) = ax_lc.plot(t_fold[0], flux[0], "o", color="k")
ax_lc.axis("on")
ax_lc.set_xlim(t_fold[0], t_fold[-1])
ax_lc.set_xlabel('Folded Phase')
ax_lc.set_ylabel('Normalised Flux')

# Plot the first frame
for i, xi, yi, zi, map, r in zip(range(1 + len(sys.secondaries)), x, y, z, maps, radii):

    # Orbit outlines
    ax_xz.plot(xi, zi)
    ax_zy.plot(zi, yi)

    # Body positions
    xz[i] = ax_xz.scatter(xi[0], zi[0])
    zy[i] = ax_zy.scatter(zi[0], yi[0])

    # Maps
    extent = np.array([xi[0], xi[0], yi[0], yi[0]]) + np.array([-1, 1, -1, 1]) * r
    xy[i] = ax.imshow(
        img[i,0],
        origin="lower",
        cmap="plasma",
        extent=extent,
        clip_on=False,
        zorder=zi[0],
    )
    circ[i] = plt.Circle(
        (xi[0], yi[0]), r, color="k", fill=False, zorder=zi[0] + 1e-3, lw=3
    )
    ax.add_artist(circ[i])

# Animation
def updatefig(k):
    for i, xi, yi, zi, map, r in zip(
        range(1 + len(sys.secondaries)), x, y, z, maps, radii
    ):
        xz[i].set_offsets((xi[k], zi[k]))
        zy[i].set_offsets((zi[k], yi[k]))
        xy[i].set_extent(
            np.array([xi[k], xi[k], yi[k], yi[k]]) + np.array([-1, 1, -1, 1]) * r
        )
        xy[i].set_zorder(zi[k])
        xy[i].set_data(img[i, k])
        circ[i].center = (xi[k], yi[k])
        circ[i].set_zorder(zi[k] + 1e-3)
    lc.set_xdata(t_fold[k])
    lc.set_ydata(flux[k])
    return xz + xy + zy + circ + [lc]


ani = animation.FuncAnimation(fig, updatefig, interval=30, blit=False, frames=len(t))
plt.close()
display(HTML(ani.to_html5_video()))

writergif = animation.FFMpegWriter(fps=20) 
ani.save('system.mp4', writer=writergif)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.lines import Line2D
from IPython.display import HTML, display


# ============================================================
# PALETA
# ============================================================

BG      = "#F7F4EE"   # fondo cálido
GREEN   = "#273A2F"   # verde oscuro
GREEN2  = "#4C6A4A"   # verde secundario
ORANGE  = "#B07D4E"   # naranja/cobre
CREAM   = "#E9D8BF"   # crema
RED     = "#8A3F2B"   # rojo terracota


# ============================================================
# FIGURA Y DISTRIBUCIÓN
# ============================================================

fig = plt.figure(figsize=(7.0, 7.8), facecolor=BG)

# Vista principal (Top)
ax = fig.add_axes([0.08, 0.31, 0.84, 0.52])

# Vista XZ
ax_xz = fig.add_axes([0.24, 0.83, 0.20, 0.15])

# Vista ZY
ax_zy = fig.add_axes([0.56, 0.83, 0.20, 0.15])

# Curva de luz
ax_lc = fig.add_axes([0.12, 0.07, 0.76, 0.17])


# ============================================================
# CONFIGURACIÓN DE LOS PANELES ESPACIALES
# ============================================================

for axis, arrs in zip(
    [ax, ax_xz, ax_zy],
    [(x, y), (x, z), (z, y)]
):

    axis.set_facecolor(BG)
    axis.axis("off")

    R = 1.2 * max(
        -np.min(arrs),
        np.max(arrs)
    )

    axis.set_xlim(-R, R)
    axis.set_ylim(-R, R)
    axis.set_aspect("equal")


# Etiquetas de las vistas
ax_xz.text(
    0.02, 0.95,
    "Side",
    transform=ax_xz.transAxes,
    fontsize=11,
    fontweight="bold",
    color=GREEN,
    ha="left",
    va="top"
)

ax_zy.text(
    0.02, 0.95,
    "Orbit",
    transform=ax_zy.transAxes,
    fontsize=11,
    fontweight="bold",
    color=GREEN,
    ha="left",
    va="top"
)


# ============================================================
# CURVA DE LUZ
# ============================================================

ax_lc.set_facecolor(BG)

ax_lc.plot(
    t_fold,
    flux,
    color=GREEN,
    lw=1.4,
    alpha=0.9,
    zorder=1
)

# Punto que seguirá la animación
(lc,) = ax_lc.plot(
    t_fold[0],
    flux[0],
    marker="o",
    markersize=6,
    markerfacecolor=ORANGE,
    markeredgecolor=GREEN,
    markeredgewidth=1.2,
    linestyle="None",
    zorder=5
)

# Línea vertical para indicar el instante actual
phase_line = ax_lc.axvline(
    t_fold[0],
    color=ORANGE,
    lw=1.2,
    ls="--",
    alpha=0.75,
    zorder=2
)

# Límites
ax_lc.set_xlim(
    np.min(t_fold),
    np.max(t_fold)
)

# Etiquetas
ax_lc.set_xlabel(
    "Folded Phase",
    fontsize=11,
    color=GREEN,
    labelpad=7
)

ax_lc.set_ylabel(
    "Normalised Flux",
    fontsize=11,
    color=GREEN,
    labelpad=7
)

# Ticks
ax_lc.tick_params(
    axis="both",
    labelsize=9,
    colors=GREEN,
    length=4
)

# Grid muy sutil
ax_lc.grid(
    True,
    alpha=0.18,
    linewidth=0.7
)

# Spines
for spine in ax_lc.spines.values():
    spine.set_color(GREEN)
    spine.set_linewidth(0.8)


# ============================================================
# OBJETOS ANIMADOS
# ============================================================

xz = [None] * (1 + len(sys.secondaries))
xy = [None] * (1 + len(sys.secondaries))
zy = [None] * (1 + len(sys.secondaries))
circ = [None] * (1 + len(sys.secondaries))

maps = (
    [sys.primary.map]
    + [sec.map for sec in sys.secondaries]
)

radii = np.array(
    [sys.primary.r]
    + [sec.r for sec in sys.secondaries]
)


# ============================================================
# COLORES DE LOS COMPONENTES
# ============================================================

# Primaria = naranja
# Secundarias = verde
body_colors = [ORANGE] + [
    GREEN2 for _ in sys.secondaries
]


# ============================================================
# PRIMER FRAME
# ============================================================

for i, xi, yi, zi, map_, r in zip(
    range(1 + len(sys.secondaries)),
    x,
    y,
    z,
    maps,
    radii
):

    # --------------------------------------------------------
    # ÓRBITAS
    # --------------------------------------------------------

    ax_xz.plot(
        xi,
        zi,
        color=GREEN2,
        lw=1.0,
        alpha=0.35
    )

    ax_zy.plot(
        zi,
        yi,
        color=GREEN2,
        lw=1.0,
        alpha=0.35
    )


    # --------------------------------------------------------
    # POSICIONES
    # --------------------------------------------------------

    xz[i] = ax_xz.scatter(
        xi[0],
        zi[0],
        s=35 if i == 0 else 22,
        color=body_colors[i],
        edgecolor=GREEN,
        linewidth=0.8,
        zorder=10
    )

    zy[i] = ax_zy.scatter(
        zi[0],
        yi[0],
        s=35 if i == 0 else 22,
        color=body_colors[i],
        edgecolor=GREEN,
        linewidth=0.8,
        zorder=10
    )


    # --------------------------------------------------------
    # MAPA DE LA ESTRELLA
    # --------------------------------------------------------

    extent = (
        np.array([
            xi[0],
            xi[0],
            yi[0],
            yi[0]
        ])
        +
        np.array([
            -1,
             1,
            -1,
             1
        ]) * r
    )

    xy[i] = ax.imshow(
        img[i, 0],
        origin="lower",

        # Mucho más suave visualmente
        cmap="magma",

        extent=extent,

        interpolation="bilinear",

        clip_on=False,

        zorder=zi[0]
    )


    # --------------------------------------------------------
    # CONTORNO DE LA ESTRELLA
    # --------------------------------------------------------

    circ[i] = plt.Circle(
        (xi[0], yi[0]),
        r,

        fill=False,

        color=body_colors[i],

        lw=1.8,

        alpha=0.95,

        zorder=zi[0] + 1e-3
    )

    ax.add_artist(circ[i])


# ============================================================
# ESTRELLA PRIMARIA: PEQUEÑO HALO
# ============================================================

primary_halo = plt.Circle(
    (x[0][0], y[0][0]),
    radii[0] * 1.08,

    fill=False,

    color=ORANGE,

    lw=0.8,

    ls="--",

    alpha=0.5,

    zorder=zi[0] + 2
)

ax.add_artist(primary_halo)


# ============================================================
# LEYENDA
# ============================================================

legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=ORANGE,
        markeredgecolor=GREEN,
        markersize=7,
        label="Primary"
    ),

    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=GREEN2,
        markeredgecolor=GREEN,
        markersize=6,
        label="Secondary"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper right",
    frameon=False,
    fontsize=9,
    labelcolor=GREEN
)


# ============================================================
# TÍTULO
# ============================================================

fig.text(
    0.5,
    0.965,
    "Eclipsing Binary Model",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color=GREEN
)


# ============================================================
# FUNCIÓN DE ANIMACIÓN
# ============================================================

def updatefig(k):

    artists = []

    for i, xi, yi, zi, map_, r in zip(
        range(1 + len(sys.secondaries)),
        x,
        y,
        z,
        maps,
        radii
    ):

        # ----------------------------------------------------
        # Posiciones en vistas orbitales
        # ----------------------------------------------------

        xz[i].set_offsets(
            (xi[k], zi[k])
        )

        zy[i].set_offsets(
            (zi[k], yi[k])
        )


        # ----------------------------------------------------
        # Mapa
        # ----------------------------------------------------

        xy[i].set_extent(
            np.array([
                xi[k],
                xi[k],
                yi[k],
                yi[k]
            ])
            +
            np.array([
                -1,
                 1,
                -1,
                 1
            ]) * r
        )

        xy[i].set_zorder(
            zi[k]
        )

        xy[i].set_data(
            img[i, k]
        )


        # ----------------------------------------------------
        # Contorno
        # ----------------------------------------------------

        circ[i].center = (
            xi[k],
            yi[k]
        )

        circ[i].set_zorder(
            zi[k] + 1e-3
        )


        artists.extend([
            xz[i],
            xy[i],
            zy[i],
            circ[i]
        ])


    # --------------------------------------------------------
    # Halo de la primaria
    # --------------------------------------------------------

    primary_halo.center = (
        x[0][k],
        y[0][k]
    )

    artists.append(primary_halo)


    # --------------------------------------------------------
    # Curva de luz
    # --------------------------------------------------------

    lc.set_xdata(
        [t_fold[k]]
    )

    lc.set_ydata(
        [flux[k]]
    )

    phase_line.set_xdata(
        [t_fold[k], t_fold[k]]
    )

    artists.extend([
        lc,
        phase_line
    ])

    return artists


# ============================================================
# CREAR ANIMACIÓN
# ============================================================

ani = animation.FuncAnimation(
    fig,
    updatefig,

    interval=30,

    frames=len(t),

    blit=False
)


plt.close()

display(
    HTML(
        ani.to_html5_video()
    )
)


# ============================================================
# GUARDAR
# ============================================================

writergif = animation.FFMpegWriter(
    fps=20
)

ani.save(
    "system.mp4",
    writer=writergif,
    dpi=200
)

## Gráfica de curvas de luz

In [ ]:
file_path = "/home/luciam/Documents/starry/Resultados_Orion/Light_Curve_Files/tic 50897998_S_6_LC.txt"

data = pd.read_csv(
    file_path,
    sep=r"\s+"
)

print(data.columns)

time = np.asarray(data["#Time"])
flux = np.asarray(data["Flux_CBV_SS"])
flat_flux = flat_flux
trend_flux = trend_flux


In [ ]:
po = p_b_50
to1 = t0_b_50

t_fold_q, q50 = phase_fold(Q50['time'].values, Q50['flux'].values, po, to1)
_, q16 = phase_fold(Q16['time'].values, Q16['flux'].values, po, to1)
_, q84 = phase_fold(Q84['time'].values, Q84['flux'].values, po, to1)

inds = t_fold_q.argsort()
t_fold_q_sorted = t_fold_q[inds]
q50_sorted = q50[inds]
q16_sorted = q16[inds]
q84_sorted = q84[inds]

print(t_fold_q_sorted)

In [ ]:
Times_norm_1 = time
Flux_norm_1 = flat_flux
ph_norm, f1_norm = phase_fold(np.array(Times_norm_1), np.array(Flux_norm_1), po, to1)

plt.figure(figsize=(10,5))
plt.scatter(ph_norm-1, f1_norm, marker="o", color='C1', s=20, alpha=0.8, edgecolors='black', lw=0.4, label='Data')
plt.plot(t_fold_q_sorted-1, q50_sorted, color='darkblue', label='Best Fit')
plt.xlabel('Folded Phase')
plt.ylabel('Normalized Flux')
plt.title('The period of the object is: {:.4f} days'.format(p_b_50))
plt.xlim(-0.5, 0.5)
#plt.ylim(min(f1_norm)*0.99, max(f1_norm)*0.97)
plt.legend(loc='lower left')



In [ ]:
print(len(time), len(flux))

In [ ]:
from matplotlib.gridspec import GridSpec
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 14})


# =========================
# FIGURA
# =========================
fig = plt.figure(figsize=(8,12), constrained_layout=True)
gs = GridSpec(3, 1, height_ratios=[1,1,1], hspace=0.06, figure=fig)

# =========================
# 1. ORIGINAL + TREND
# =========================
ax1 = fig.add_subplot(gs[0])

ax1.scatter(time, flux,
            s=10, c='black',
            label='Original')

ax1.plot(time, trend_flux,
         c='r', lw=2, ls='--',
         label='Trend')

ax1.set_ylabel("Flux")
ax1.set_ylim(np.min(flux)*0.995, np.max(flux)*1.006)
ax1.legend(loc='upper left')


# =========================
# 2. FLATTENED
# =========================
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax2.scatter(time, flat_flux,
            c='mediumblue', s=5,
            label='Flattened')

ax2.set_ylabel("Normalized Flux")
ax2.set_xlabel("Time [days]")
ax2.set_ylim(np.min(flat_flux)*0.995, np.max(flat_flux)*1.006)
ax2.legend(loc='upper left')


# =========================
# 3. PHASE FOLD
# =========================

# centrar fase correctamente 👀
phase_data = ph_norm - np.floor(ph_norm + 0.5)
phase_model = t_fold_q_sorted - np.floor(t_fold_q_sorted + 0.5)

# ordenar modelo para evitar artefactos
inds = np.argsort(phase_model)
phase_model = phase_model[inds]
q50_sorted = q50_sorted[inds]
q16_sorted = q16_sorted[inds]
q84_sorted = q84_sorted[inds]

ax3 = fig.add_subplot(gs[2])

# DATA
ax3.scatter(phase_data, f1_norm,
            s=20, color='C1',
            alpha=0.8,
            edgecolors='black', lw=0.4,
            label='Data')

# MODELO (sin línea rara 👀)
ax3.scatter(phase_model, q50_sorted,
            color='darkblue', s=5,
            label='Best Fit')

# POSTERIOR
ax3.fill_between(phase_model,
                 q16_sorted, q84_sorted,
                 alpha=0.3, color='darkblue',
                 label='Posterior')

ax3.set_xlabel('Phase')
ax3.set_ylabel('Normalized Flux')
ax3.set_xlim(-0.5, 0.5)

ax3.legend(loc='lower right')

ax3.text(-0.48, np.min(q50_sorted),
         f'$P$ = {po:.4f} days',
         fontsize=14)

# =========================
# SAVE
# =========================
filename = f"Full_{system.split('/')[-1]}.png"
plt.savefig(filename, dpi=300)

plt.show()

In [ ]:
# Get the posterior median orbital parameters
M_a_16, M_a_50, M_a_84         = np.percentile(Pos['A_m'], [16, 50, 84])
R_a_16, R_a_50, R_a_84         = np.percentile(Pos['A_r'], [16, 50, 84])
i_a_16, i_a_50, i_a_84         = np.percentile(Pos['A_inc'], [16, 50, 84])
amp_a_16, amp_a_50, amp_a_84   = np.percentile(Pos['A_amp'], [16, 50, 84])
u0_a_16, u0_a_50, u0_a_84      = np.percentile(Pos['A_u0'], [16, 50, 84])
u1_a_16, u1_a_50, u1_a_84      = np.percentile(Pos['A_u1'], [16, 50, 84])

M_b_16, M_b_50, M_b_84         = np.percentile(Pos['B_m'], [16, 50, 84])
R_b_16, R_b_50, R_b_84         = np.percentile(Pos['B_r'], [16, 50, 84])
i_b_16, i_b_50, i_b_84         = np.percentile(Pos['B_inc'], [16, 50, 84])
p_b_16, p_b_50, p_b_84         = np.percentile(Pos['B_porb'], [16, 50, 84])
t0_b_16, t0_b_50, t0_b_84      = np.percentile(Pos['B_t0'], [16, 50, 84])
e_b_16, e_b_50, e_b_84         = np.percentile(Pos['B_ecc'], [16, 50, 84])
w_b_16, w_b_50, w_b_84         = np.percentile(Pos['B_w'], [16, 50, 84])
amp_b_16, amp_b_50, amp_b_84   = np.percentile(Pos['B_amp'], [16, 50, 84])
u0_b_16, u0_b_50, u0_b_84      = np.percentile(Pos['B_u0'], [16, 50, 84])
u1_b_16, u1_b_50, u1_b_84      = np.percentile(Pos['B_u1'], [16, 50, 84])

a_b_16, a_b_50, a_b_84         = np.percentile(a, [16, 50, 84])

In [ ]:
data_A = {
    'Variable' : ['$M_A$', '$R_A$', '$u_{0,A}$', '$u_{1,A}$'],
    'Median'    : [M_a_50, R_a_50, u0_a_50, u1_a_50],
    'Q16'       : [M_a_16-M_a_50, R_a_16-R_a_50, u0_a_16-u0_a_50, u1_a_16-u1_a_50],
    'Q86'       : [M_a_84-M_a_50, R_a_84-R_a_50, u0_a_84-u0_a_50, u1_a_84-u1_a_50],
}

Final_A = pd.DataFrame(data_A)
display(Final_A.style.hide())



data_B = {
    'Variable' : ['$M_B$', '$R_B$', '$i$', '$P_o$', '$a$', '$t_0$', '$e$', '$\omega$', '$u_{0,B}$', '$u_{1,B}$'],
    'Median'    : [M_b_50, R_b_50, i_b_50, p_b_50, a_b_50, t0_b_50, e_b_50, w_b_50, u0_b_50, u1_b_50],
    'Q16'   : [M_b_16-M_b_50, R_b_16-R_b_50, i_b_16-i_b_50, p_b_16-p_b_50, 
                   a_b_16-a_b_50, t0_b_16-t0_b_50, e_b_16-e_b_50, w_b_16-w_b_50, u0_b_16-u0_b_50, u1_b_16-u1_b_50],
    'Q86'   : [M_b_84-M_b_50, R_b_84-R_b_50, i_b_84-i_b_50, p_b_84-p_b_50, 
                   a_b_84-a_b_50, t0_b_84-t0_b_50, e_b_84-e_b_50, w_b_84-w_b_50, u0_b_84-u0_b_50, u1_b_84-u1_b_50],
}

Final_B = pd.DataFrame(data_B)
display(Final_B.style.hide())


In [ ]:
Fl_Det = flux/trend_flux

fig = plt.figure(figsize=(12,5), dpi=300)
#plt.plot(Time, Flux, ".k")
#plt.plot(Times, Fl_Det, c='b')
plt.scatter(time, Fl_Det, c='b',s = 10)#s tamaño de punto 
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
from scipy.stats import median_abs_deviation

P_orb = 2.903
t0 = to1   # tiempo de tránsito (ya lo tienes)

phase_orb = ((time - t0) / P_orb) % 1
phase_orb = phase_orb - 0.5

mask = (np.abs(phase_orb) > 0.15) & (np.abs(np.abs(phase_orb) - 0.5) > 0.1)

time_clean = time[mask]
flux_clean = (flux / trend_flux)[mask]   

plt.figure(figsize=(10,4))

plt.scatter(time, flux/trend_flux,
            s=5, color='gray', alpha=0.3,
            label='Original')

plt.scatter(time_clean, flux_clean,
            s=5, color='blue',
            label='Sin eclipses')

plt.xlabel("Time [days]")
plt.ylabel("Normalized Flux")
plt.legend()

plt.title("Curva sin eclipses")
plt.tight_layout()
plt.show()

time_cont = np.arange(len(time_clean))

plt.figure(figsize=(10,4))

plt.scatter(time_cont, flux_clean,
            s=5, color='blue')

plt.xlabel("Time index (continuous)")
plt.ylabel("Normalized Flux")
plt.title("Curva sin eclipses (continua)")

plt.tight_layout()
plt.show()


In [ ]:

# =========================
# 1. NORMALIZAR
# =========================
flux_ls = flux_clean / np.median(flux_clean)
flux_ls = flux_ls - np.mean(flux_ls)

# =========================
# 2. LOMB–SCARGLE
# =========================
ls = LombScargle(time_clean, flux_ls)

frequency, power = ls.autopower(
    minimum_frequency=0.1,   # periodos largos (~10 d)
    maximum_frequency=5.0    # periodos cortos (~0.2 d)
)

period = 1 / frequency

# =========================
# 3. MEJOR PERIODO
# =========================
best_period = period[np.argmax(power)]

print(f"Periodo dominante: {best_period:.4f} días")
print(f"Posible rotación: {2*best_period:.4f} días (si es P/2)")

# =========================
# 4. PLOT PERIODOGRAMA
# =========================
plt.figure(figsize=(10,4))

plt.plot(period, power, 'k-')

# marcar P/2
plt.axvline(best_period,
            color='red', ls='--',
            label=f'{best_period:.3f} d (P/2?)')

# marcar P
plt.axvline(2*best_period,
            color='blue', ls='--',
            label=f'{2*best_period:.3f} d (P?)')

plt.xlabel("Period [days]")
plt.ylabel("Power")
plt.xlim(0, 5)
plt.legend()

plt.title("Lomb-Scargle (sin eclipses)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# periodo (elige cuál quieres)
# =========================
P_plot = 2* best_period   # o best_period si quieres probar P/2

# =========================
# t0 (puedes usar orbital o rotacional)
# =========================
t0_plot = to1   # o to1 si quieres eclipse centrado

# =========================
# fase en [0,1]
# =========================
phase = ((time_clean - t0_plot) / P_plot) % 1

# =========================
# ordenar (IMPORTANTE 👀)
# =========================
inds = np.argsort(phase)
phase = phase[inds]
flux_plot = flux_clean[inds]

# =========================
# plot estilo transit
# =========================
plt.figure(figsize=(6,4))

plt.scatter(phase, flux_plot,
            s=10, color='red', alpha=0.4)

plt.xlabel("Phase [days]")
plt.ylabel("Flux Corrected")

plt.xlim(0, 1)

plt.tight_layout()
plt.show()